# Powering our chatbot with tools

In [1]:
from dotenv import load_dotenv
import os

## Setup API Keys

In [2]:
load_dotenv()
SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")

assert SARVAM_API_KEY is not None, "Could NOT load SARVAM_API_KEY from .env"

## Use Case Definition

Build a chatbot for ABC Bank

The chatbot interacts with customers online helping them:
1. Check the Account Balance
2. Review the list of transactions in their account in the last 24 hours

## Setup the tools

In [3]:
def get_balance(account_number: str):
    if account_number == '001002':
        return "INR 51203"
    else:
        return "INR 2500"

def get_transactions(account_number: str):
    if account_number == "001002":
        return [
            ("Time", "Transaction", "INR"),
            ("11:21 AM", "Debit - XYZ Super Market", "INR 651.50"),
            ("4:45 PM",  "Credit - Refund from PQR Braodband Services", "INR 999"),
        ]
    else:
        return "No transactions in the last 24 hrs in your account"

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_balance",
            "description": "Get the balance for the account number",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },
     {
        "type": "function",
        "function": {
            "name": "get_transactions",
            "description": "Get transactions from the last 24 hours",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },   
]

In [5]:
tools_map  = {
    "get_balance": get_balance,
    "get_transactions": get_transactions,
}

def invoke_tool(f_name: str, f_args: dict):
    return tools_map[f_name](**f_args)

### Let us test our tools

In [6]:
invoke_tool("get_balance", {"account_number": "001002"})

'INR 51203'

In [7]:
invoke_tool("get_transactions", {"account_number": "001002"})

[('Time', 'Transaction', 'INR'),
 ('11:21 AM', 'Debit - XYZ Super Market', 'INR 651.50'),
 ('4:45 PM', 'Credit - Refund from PQR Braodband Services', 'INR 999')]

### Give the chatbot access to the tools

In [8]:
from sarvamai import SarvamAI

In [9]:
client = SarvamAI(
    api_subscription_key=SARVAM_API_KEY,
)

In [10]:
def chat(messages: list):
    response = client.chat.completions(
        model="sarvam-105b-conversations",
        messages=messages,
        tools=tools,
    )    
    print(response.model_dump_json(indent=2))
    return response.choices[0].message.content

#### Send a message to the chatbot

In [11]:
messages=[
    {"role": "system", "content": "You are a Customer Service Rep from ABC Bank."},
    {"role": "user", "content": "Hi, I am Amit. How are you?"},
]

chatbot_response = chat(messages)

{
  "id": "20260925_11288221-e061-48d5-a3c8-4b46ec66dc88",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Hi Amit! I'm doing well, thank you for asking. How can I assist you today with your ABC Bank account?",
        "role": "assistant",
        "refusal": null,
        "reasoning_content": null,
        "tool_calls": null
      },
      "logprobs": null
    }
  ],
  "created": 1790356786,
  "model": "sarvam-105b-conversations",
  "object": "chat.completion",
  "system_fingerprint": "vllm-0.29.1rc1.dev145+gfe61f3317-9c902b0a",
  "usage": {
    "completion_tokens": 27,
    "prompt_tokens": 189,
    "total_tokens": 216,
    "completion_tokens_details": null,
    "prompt_tokens_details": null
  },
  "service_tier": null
}


In [12]:
print(chatbot_response)

Hi Amit! I'm doing well, thank you for asking. How can I assist you today with your ABC Bank account?


#### Send a message requiring a tool to answer

In [14]:
messages=[
    {"role": "system", "content": "You are a Customer Service Rep from ABC Bank."},
    {"role": "user", "content": "hi, what is the balance of accounts 001002?"},
]

chatbot_response = chat(messages)

{
  "id": "20260925_1b73f5cc-0df3-4ef4-97c2-16d6c2f979c8",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "message": {
        "role": "assistant",
        "tool_calls": [
          {
            "id": "chatcmpl-tool-abf788f80f63a62f",
            "type": "function",
            "function": {
              "name": "get_balance",
              "arguments": "{\"account_number\": \"001002\"}"
            }
          }
        ],
        "content": null,
        "refusal": null,
        "reasoning_content": null
      },
      "logprobs": null
    }
  ],
  "created": 1790356817,
  "model": "sarvam-105b-conversations",
  "object": "chat.completion",
  "system_fingerprint": "vllm-0.29.1rc1.dev145+gfe61f3317-9c902b0a",
  "usage": {
    "completion_tokens": 22,
    "prompt_tokens": 195,
    "total_tokens": 217,
    "completion_tokens_details": null,
    "prompt_tokens_details": null
  },
  "service_tier": null
}


In [20]:
print(chatbot_response)

None


## Integrate the chatbot with tools

In [21]:
import json

BLUE_TEXT = "\033[94m"
BLACK_TEXT = "\033[0m"

In [22]:
def run_chatbot():

    messages=[
        {"role": "system", "content": "You are a Customer Service Rep from ABC Bank."},
    ]
    
    while True:
        user_message = input("\nUser:")
        
        if user_message == "quit":
            break
            
        messages.append(
            {"role": "user", "content": user_message.strip()}
        )

        while True:
            response = client.chat.completions(
                model="sarvam-105b-conversations",
                messages=messages,
                tools=tools,
            )
            if not response.choices[0].message.tool_calls:
                # No tool calls - Let us display the chatbot response to the user
                break
                
            tool_calls = response.choices[0].message.tool_calls
            
            for tool_call in tool_calls:
                f_name = tool_call.function.name #get_balance
                f_args = json.loads(tool_call.function.arguments) # {"account_number": "001002"}
                result = invoke_tool(f_name, f_args)

                messages.append(
                    {
                        "role": "assistant",
                        "tool_calls": [
                            {
                                "id": tool_call.id,
                                "type": "function",
                                "function": {
                                    "name": tool_call.function.name,
                                    "arguments": tool_call.function.arguments,
                                },
                            }
                        ],
                    }
                )
                    
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": str(result),
                    }
                )             

        print("\n", BLUE_TEXT, "Assistant: ", response.choices[0].message.content.strip(), BLACK_TEXT)
        messages.append(
            {"role": "assistant", "content": response.choices[0].message.content}
        )

In [23]:
run_chatbot()


User: hi



  Assistant:  Hello! Welcome to ABC Bank. How can I assist you today? 



User: balance for 001002



  Assistant:  The balance for account **001002** is **INR 51,203**. Is there anything else I can help you with? 



User: quit
